# 04 — CICIDS2017 Preprocessing & Model Training
**NIDS — Network Intrusion Detection System**

This notebook:
1. Loads all CICIDS2017 CSVs from `data/raw/CICIDS2017/`
2. Cleans columns: strips whitespace, drops infinite/NaN, drops zero-variance features
3. Encodes labels & scales features (same `ColumnTransformer` pattern as notebook 02)
4. Applies SMOTE on minority attack classes
5. Trains XGBoost and Random Forest, selects the best
6. Saves `data/processed/cicids/` arrays and `models/cicids_model.pkl`

> **Why CICIDS2017?**  NSL-KDD (1999) misses modern attacks like web-based and infiltration threats.
> CICIDS2017 is flow-level traffic from a 2017 testbed — far closer to real-world NIDS deployment.
> Training on both gives the dashboard a realistic multi-dataset story.

## 0. Imports & Paths

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, glob, json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

import mlflow
import mlflow.xgboost
import mlflow.sklearn

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR      = os.path.dirname(os.getcwd())
RAW_CICIDS    = os.path.join(BASE_DIR, 'data', 'raw', 'CICIDS2017')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed', 'cicids')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

mlflow.set_tracking_uri(
    f'sqlite:///{os.path.join(BASE_DIR, "mlruns", "mlflow.db")}'
)

print('CICIDS2017 raw dir:', RAW_CICIDS)
csv_files = sorted(glob.glob(os.path.join(RAW_CICIDS, '*.csv')))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    print(' ', os.path.basename(f))

CICIDS2017 raw dir: /Users/adityasitapara/Desktop/NIDS/data/raw/CICIDS2017
Found 8 CSV files:
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv


In [2]:
import os, glob

BASE_DIR   = os.path.dirname(os.getcwd())
RAW_CICIDS = os.path.join(BASE_DIR, 'data', 'raw', 'CICIDS2017')

print('Path  :', RAW_CICIDS)
print('Exists:', os.path.isdir(RAW_CICIDS))

if os.path.isdir(RAW_CICIDS):
    all_files = os.listdir(RAW_CICIDS)
    print(f'\nAll contents ({len(all_files)} items):')
    for f in sorted(all_files):
        fpath = os.path.join(RAW_CICIDS, f)
        size  = os.path.getsize(fpath)
        print(f'  {f:50s}  {size/1e6:.1f} MB')
else:
    print('\nFolder does not exist.')
    print('Parent contents:', os.listdir(os.path.join(BASE_DIR, 'data', 'raw')))

Path  : /Users/adityasitapara/Desktop/NIDS/data/raw/CICIDS2017
Exists: True

All contents (8 items):
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    77.1 MB
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  76.9 MB
  Friday-WorkingHours-Morning.pcap_ISCX.csv           58.3 MB
  Monday-WorkingHours.pcap_ISCX.csv                   176.9 MB
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  83.1 MB
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  52.0 MB
  Tuesday-WorkingHours.pcap_ISCX.csv                  135.1 MB
  Wednesday-workingHours.pcap_ISCX.csv                225.2 MB


In [3]:
import os, glob

BASE_DIR   = os.path.dirname(os.getcwd())
RAW_CICIDS = os.path.join(BASE_DIR, 'data', 'raw', 'CICIDS2017')

print('Path  :', RAW_CICIDS)
print('Exists:', os.path.isdir(RAW_CICIDS))
print(f'\nAll contents ({len(os.listdir(RAW_CICIDS))} items):')
for f in sorted(os.listdir(RAW_CICIDS)):
    fpath = os.path.join(RAW_CICIDS, f)
    if os.path.isdir(fpath):
        print(f'  [DIR]  {f}  → {os.listdir(fpath)[:5]}')
    else:
        print(f'  [FILE] {f:50s}  {os.path.getsize(fpath)/1e6:.1f} MB')

print('\nGlob *.csv result:')
csv_files = glob.glob(os.path.join(RAW_CICIDS, '*.csv'))
print(f'  Found {len(csv_files)} files')

Path  : /Users/adityasitapara/Desktop/NIDS/data/raw/CICIDS2017
Exists: True

All contents (8 items):
  [FILE] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    77.1 MB
  [FILE] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  76.9 MB
  [FILE] Friday-WorkingHours-Morning.pcap_ISCX.csv           58.3 MB
  [FILE] Monday-WorkingHours.pcap_ISCX.csv                   176.9 MB
  [FILE] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  83.1 MB
  [FILE] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  52.0 MB
  [FILE] Tuesday-WorkingHours.pcap_ISCX.csv                  135.1 MB
  [FILE] Wednesday-workingHours.pcap_ISCX.csv                225.2 MB

Glob *.csv result:
  Found 8 files


## 1. Load & Merge CSVs

> CICIDS2017 is split across 8 day-files. We load them all and concatenate.

In [4]:
dfs = []
for path in csv_files:
    df_tmp = pd.read_csv(path, low_memory=False)
    dfs.append(df_tmp)
    print(f'  {os.path.basename(path):45s}  {df_tmp.shape}')

df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined shape: {df.shape}')
print(df.dtypes.value_counts())

  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  (288602, 79)
  Monday-WorkingHours.pcap_ISCX.csv              (529918, 79)
  Friday-WorkingHours-Morning.pcap_ISCX.csv      (191033, 79)
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  (286467, 79)
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  (225745, 79)
  Tuesday-WorkingHours.pcap_ISCX.csv             (445909, 79)
  Wednesday-workingHours.pcap_ISCX.csv           (692703, 79)
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  (170366, 79)

Combined shape: (2830743, 79)
int64      54
float64    24
object      1
Name: count, dtype: int64


## 2. Column Cleaning

> CICIDS2017 column names have leading/trailing spaces and sometimes special characters. Strip and normalise them.

In [5]:
# ── Strip whitespace from column names & string values ────────
df.columns = df.columns.str.strip()

# Rename the label column to a canonical name
LABEL_COL = ' Label' if ' Label' in df.columns else 'Label'
df = df.rename(columns={LABEL_COL: 'label'})

# Strip whitespace from label strings
df['label'] = df['label'].str.strip()

print('Label value counts (top 15):')
print(df['label'].value_counts().head(15))
print(f'\nTotal rows: {len(df):,}')

Label value counts (top 15):
label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Total rows: 2,830,743


In [6]:
# ── Drop non-numeric / identifier columns ──────────────────────
DROP_COLS = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp',
             'Src IP', 'Dst IP', 'Src Port', 'Dst Port',   # alt names
             'Source Port', 'Destination Port']              # alt names
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')

# ── Force numeric; coerce errors to NaN ──────────────────────
label_series = df.pop('label')
df = df.apply(pd.to_numeric, errors='coerce')
df['label'] = label_series

print('Shape after dropping ID cols:', df.shape)
print(f'NaN count: {df.isnull().sum().sum():,}')

Shape after dropping ID cols: (2830743, 78)
NaN count: 1,358


In [7]:
# ── Replace Inf with NaN, then drop rows with any NaN ─────────
df.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(df)
df.dropna(inplace=True)
after  = len(df)
print(f'Dropped {before - after:,} rows with Inf/NaN ({(before-after)/before*100:.1f}%)')
print(f'Remaining rows: {after:,}')

Dropped 2,867 rows with Inf/NaN (0.1%)
Remaining rows: 2,827,876


In [8]:
# ── Drop zero-variance features (constant columns) ────────────
feature_cols = [c for c in df.columns if c != 'label']
X_raw = df[feature_cols]
var_mask  = X_raw.var() > 0
dropped   = feature_cols_dropped = list(X_raw.columns[~var_mask])
df        = df.drop(columns=dropped)
feature_cols = [c for c in df.columns if c != 'label']
print(f'Dropped {len(dropped)} zero-variance features: {dropped}')
print(f'Remaining features: {len(feature_cols)}')

Dropped 8 zero-variance features: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Remaining features: 69


## 3. Label Consolidation & Encoding

> CICIDS2017 has fine-grained attack sub-labels (e.g. 'DoS Hulk', 'DoS GoldenEye').  
> We keep them as-is so the model learns sub-type distinctions, but print the distribution.

In [9]:
# ── Optionally remap fine-grained to coarse labels ────────────
# Comment this block out to keep sub-type labels.
LABEL_MAP = {
    'BENIGN'               : 'Benign',
    'DoS Hulk'             : 'DoS',
    'DoS GoldenEye'        : 'DoS',
    'DoS slowloris'        : 'DoS',
    'DoS Slowhttptest'     : 'DoS',
    'Heartbleed'           : 'DoS',
    'DDoS'                 : 'DDoS',
    'PortScan'             : 'PortScan',
    'FTP-Patator'          : 'BruteForce',
    'SSH-Patator'          : 'BruteForce',
    'Bot'                  : 'Bot',
    'Web Attack – Brute Force' : 'WebAttack',
    'Web Attack – XSS'         : 'WebAttack',
    'Web Attack – Sql Injection': 'WebAttack',
    'Infiltration'         : 'Infiltration',
}
df['label'] = df['label'].map(LABEL_MAP).fillna(df['label'])

print('Label distribution after remapping:')
vc = df['label'].value_counts()
print(vc)
CLASS_NAMES = list(vc.index)

Label distribution after remapping:
label
Benign                        2271320
DoS                            251723
PortScan                       158804
DDoS                           128025
BruteForce                      13832
Bot                              1956
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Name: count, dtype: int64


In [10]:
# ── Encode labels ─────────────────────────────────────────────
le = LabelEncoder()
y_all = le.fit_transform(df['label'])
CLASS_NAMES = list(le.classes_)
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

# ── Save encoder ───────────────────────────────────────────────
joblib.dump(le, os.path.join(PROCESSED_DIR, 'target_encoder.pkl'))
print('✓ target_encoder.pkl saved')

Classes (10): ['Benign', 'Bot', 'BruteForce', 'DDoS', 'DoS', 'Infiltration', 'PortScan', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']
✓ target_encoder.pkl saved


## 4. Feature Scaling & Train/Val/Test Split

In [11]:
X_all = df[feature_cols].values.astype(np.float32)

# ── Scale features ─────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
joblib.dump(scaler,       os.path.join(PROCESSED_DIR, 'scaler.pkl'))
joblib.dump(feature_cols, os.path.join(PROCESSED_DIR, 'feature_names.pkl'))

# ── Stratified split: 70 % train / 15 % val / 15 % test ───────
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_scaled, y_all, test_size=0.30, random_state=42, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

print(f'X_train : {X_tr.shape}   y_train : {y_tr.shape}')
print(f'X_val   : {X_val.shape}   y_val   : {y_val.shape}')
print(f'X_test  : {X_test.shape}   y_test  : {y_test.shape}')

X_train : (1979513, 69)   y_train : (1979513,)
X_val   : (424181, 69)   y_val   : (424181,)
X_test  : (424182, 69)   y_test  : (424182,)


## 5. SMOTE — Handle Class Imbalance

> Rare classes (Infiltration, Heartbleed, WebAttack) have very few samples.  
> We apply SMOTE only on training data to avoid data leakage.

In [12]:
# Compute per-class count; only SMOTE if we have enough samples
from collections import Counter
class_counts = Counter(y_tr)
MIN_SAMPLES  = 10   # SMOTE requires k_neighbors+1 samples per class

# Classes too small for default k=5 — use k=min(count-1, 5)
k_neighbors = min(min(class_counts.values()) - 1, 5)
k_neighbors = max(k_neighbors, 1)

print(f'Class distribution before SMOTE: {dict(sorted(class_counts.items()))}')
print(f'Using k_neighbors={k_neighbors}')

smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
X_train, y_train = smote.fit_resample(X_tr, y_tr)
print(f'Shape after SMOTE: {X_train.shape}')
print(f'Class distribution after SMOTE: {dict(Counter(y_train))}')

Class distribution before SMOTE: {np.int64(0): 1589924, np.int64(1): 1369, np.int64(2): 9682, np.int64(3): 89618, np.int64(4): 176206, np.int64(5): 25, np.int64(6): 111163, np.int64(7): 1055, np.int64(8): 15, np.int64(9): 456}
Using k_neighbors=5
Shape after SMOTE: (15899240, 69)
Class distribution after SMOTE: {np.int64(0): 1589924, np.int64(3): 1589924, np.int64(4): 1589924, np.int64(6): 1589924, np.int64(2): 1589924, np.int64(7): 1589924, np.int64(9): 1589924, np.int64(1): 1589924, np.int64(8): 1589924, np.int64(5): 1589924}


In [13]:
# ── Save processed arrays ─────────────────────────────────────
for name, arr in [('X_train', X_train), ('y_train', y_train),
                   ('X_val',   X_val),   ('y_val',   y_val),
                   ('X_test',  X_test),  ('y_test',  y_test)]:
    np.save(os.path.join(PROCESSED_DIR, f'{name}.npy'), arr)
    print(f'  Saved {name}.npy')
print('✓ All processed arrays saved')

  Saved X_train.npy
  Saved y_train.npy
  Saved X_val.npy
  Saved y_val.npy
  Saved X_test.npy
  Saved y_test.npy
✓ All processed arrays saved


## 6. Train & Compare Models

In [ ]:
_artifact_root = f'file://{os.path.join(BASE_DIR, "mlruns", "artifacts")}'

# ── Tuned XGBoost ──────────────────────────────────────────────
exp_name = 'NIDS-CICIDS-XGBoost'
if mlflow.get_experiment_by_name(exp_name) is None:
    mlflow.create_experiment(exp_name, artifact_location=_artifact_root)
mlflow.set_experiment(exp_name)

xgb_params = dict(
    n_estimators          = 200,
    max_depth             = 5,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.75,
    min_child_weight      = 8,
    gamma                 = 0.2,
    reg_alpha             = 0.1,
    reg_lambda            = 1.5,
    objective             = 'multi:softprob',
    num_class             = len(CLASS_NAMES),
    eval_metric           = 'mlogloss',
    early_stopping_rounds = 20,
    random_state          = 42,
    n_jobs                = -1,
    tree_method           = 'hist',
)

with mlflow.start_run(run_name='xgb_cicids') as run_xgb:
    mlflow.log_params(xgb_params)
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=25)

    val_acc_x  = accuracy_score(y_val,  xgb_model.predict(X_val))
    test_acc_x = accuracy_score(y_test, xgb_model.predict(X_test))
    test_f1_x  = f1_score(y_test, xgb_model.predict(X_test), average='weighted')
    mlflow.log_metrics({'val_accuracy': val_acc_x, 'test_accuracy': test_acc_x, 'test_f1_weighted': test_f1_x})
    RUN_ID_XGB = run_xgb.info.run_id

print(f'[XGBoost]  Val: {val_acc_x:.4f}  Test: {test_acc_x:.4f}  F1: {test_f1_x:.4f}')

[0]	validation_0-mlogloss:2.02417
[25]	validation_0-mlogloss:0.41438
[50]	validation_0-mlogloss:0.14085
[75]	validation_0-mlogloss:0.07266
[99]	validation_0-mlogloss:0.04821
[XGBoost]  Val: 0.9854  Test: 0.9854  F1: 0.9896


In [ ]:
# ── Random Forest ─────────────────────────────────────────────
exp_name_rf = 'NIDS-CICIDS-RandomForest'
if mlflow.get_experiment_by_name(exp_name_rf) is None:
    mlflow.create_experiment(exp_name_rf, artifact_location=_artifact_root)
mlflow.set_experiment(exp_name_rf)

rf_params = dict(
    n_estimators     = 200,
    max_depth        = 25,
    min_samples_leaf = 4,
    max_features     = 'sqrt',
    class_weight     = 'balanced',
    random_state     = 42,
    n_jobs           = -1,
)

with mlflow.start_run(run_name='rf_cicids') as run_rf:
    mlflow.log_params(rf_params)
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train, y_train)

    val_acc_rf  = accuracy_score(y_val,  rf_model.predict(X_val))
    test_acc_rf = accuracy_score(y_test, rf_model.predict(X_test))
    test_f1_rf  = f1_score(y_test, rf_model.predict(X_test), average='weighted')
    mlflow.log_metrics({'val_accuracy': val_acc_rf, 'test_accuracy': test_acc_rf, 'test_f1_weighted': test_f1_rf})
    RUN_ID_RF = run_rf.info.run_id

print(f'[RandomForest]  Val: {val_acc_rf:.4f}  Test: {test_acc_rf:.4f}  F1: {test_f1_rf:.4f}')

In [ ]:
# ── Select best ───────────────────────────────────────────────
results = pd.DataFrame([
    {'Model': 'XGBoost',       'Val Acc': val_acc_x,  'Test Acc': test_acc_x,  'Test F1 (w)': test_f1_x},
    {'Model': 'Random Forest', 'Val Acc': val_acc_rf, 'Test Acc': test_acc_rf, 'Test F1 (w)': test_f1_rf},
]).set_index('Model')
print(results.round(4).to_string())

if test_f1_x >= test_f1_rf:
    best_name, model, params, RUN_ID, model_type = 'XGBoost', xgb_model, xgb_params, RUN_ID_XGB, 'xgboost'
else:
    best_name, model, params, RUN_ID, model_type = 'Random Forest', rf_model, rf_params, RUN_ID_RF, 'sklearn'

y_val_pred  = model.predict(X_val)
y_test_pred = model.predict(X_test)
val_acc     = accuracy_score(y_val,  y_val_pred)
val_f1      = f1_score(y_val,  y_val_pred,  average='weighted')
test_acc    = accuracy_score(y_test, y_test_pred)
test_f1     = f1_score(y_test, y_test_pred, average='weighted')

print(f'\n✓ Selected: {best_name}  |  Test Acc: {test_acc:.4f}  |  Test F1: {test_f1:.4f}')

## 7. Evaluation — Classification Report

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
print('=' * 60)
print(f'  TEST SET — Classification Report  [{best_name}]')
print('=' * 60)
print(classification_report(y_test, y_test_pred, target_names=CLASS_NAMES))

## 8. Confusion Matrix & Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title(f'Confusion Matrix — CICIDS2017 [{best_name}]', fontsize=14, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()

cm_path = os.path.join(MODELS_DIR, 'cicids_confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Saved → {cm_path}')

In [ ]:
importances = model.feature_importances_
top_n     = 20
top_idx   = np.argsort(importances)[::-1][:top_n]
top_names = [feature_cols[i] for i in top_idx]
top_vals  = importances[top_idx]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top_names[::-1], top_vals[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances — CICIDS2017 [{best_name}]', fontsize=13, fontweight='bold')
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
plt.tight_layout()

fi_path = os.path.join(MODELS_DIR, 'cicids_feature_importance.png')
plt.savefig(fi_path, dpi=150)
plt.show()
print(f'Saved → {fi_path}')

## 9. Save Model + Model Info

In [ ]:
model_path = os.path.join(MODELS_DIR, 'cicids_model.pkl')
joblib.dump(model, model_path)
print(f'✓ Model saved → {model_path}')

cm_dict = {
    actual_cls: {pred_cls: int(cm[i][j]) for j, pred_cls in enumerate(CLASS_NAMES)}
    for i, actual_cls in enumerate(CLASS_NAMES)
}
fi_list = [[feature_cols[i], float(importances[i])] for i in np.argsort(importances)[::-1][:20]]

model_info = {
    'algorithm'        : best_name,
    'dataset'          : 'CICIDS2017',
    'samples'          : int(len(X_train)),
    'features'         : int(len(feature_cols)),
    'train_size'       : f'{len(X_train):,} rows (post-SMOTE)',
    'test_accuracy'    : round(test_acc, 4),
    'test_f1_weighted' : round(test_f1, 4),
    'val_accuracy'     : round(val_acc, 4),
    'val_f1_weighted'  : round(val_f1, 4),
    'confusion_matrix' : cm_dict,
    'feature_importance': fi_list,
    'classes'          : CLASS_NAMES,
    'mlflow_run_id'    : RUN_ID,
    'hyperparameters'  : {k: str(v) for k, v in params.items()},
    'model_comparison' : results.reset_index().to_dict(orient='records'),
}

info_path = os.path.join(MODELS_DIR, 'cicids_model_info.json')
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
print(f'✓ Model info → {info_path}')

## 10. Log to MLflow

In [ ]:
with mlflow.start_run(run_id=RUN_ID):
    if model_type == 'xgboost':
        mlflow.xgboost.log_model(model, name='cicids_model')
    else:
        mlflow.sklearn.log_model(model, name='cicids_model')
    mlflow.log_artifact(cm_path, 'plots')
    mlflow.log_artifact(fi_path, 'plots')
    mlflow.log_artifact(info_path, 'model_info')
print('✓ Artifacts logged to MLflow')

## 11. Final Summary

In [ ]:
print('=' * 65)
print('  NIDS — CICIDS2017 Training Complete')
print('=' * 65)
print(f'  Winner     : {best_name}')
print(f'  Dataset    : CICIDS2017')
print(f'  Classes    : {CLASS_NAMES}')
print(f'  Features   : {len(feature_cols)}')
print(f'  Train rows : {len(X_train):,} (post-SMOTE)')
print()
print('  Model Comparison:')
print(results.round(4).to_string())
print()
print(f'  Val  Accuracy : {val_acc:.4f}  |  Val  F1 : {val_f1:.4f}')
print(f'  Test Accuracy : {test_acc:.4f}  |  Test F1 : {test_f1:.4f}')
print()
print(f'  cicids_model.pkl      → {model_path}')
print(f'  cicids_model_info.json → {info_path}')
print('=' * 65)
print('  Next: wire cicids_model.pkl into ml.py with a dataset= query param')
print('=' * 65)